In [ ]:
import os
import sys
import warnings

warnings.filterwarnings("ignore")
sys.path.append(os.path.abspath('..'))

import time
import numpy as np
import pandas as pd
import kagglehub
from sklearn.model_selection import train_test_split

import torch
import torch.optim as optim
import torch.optim.lr_scheduler as lr_scheduler
from torch.utils.data import DataLoader, SequentialSampler

import albumentations as A
from albumentations.pytorch import ToTensorV2

from utils.datasets import BusbraDataset
from utils.metric import BCEDiceLoss
from utils.visualization import predict_compare, plot_history_loss
from utils.training import train
from models.UNet import BaseUNet

# Hyper-parameters

In [ ]:
IMG_SIZE = (256, 256)
BATCH_SIZE = 8
NUM_WORKERS = 2
NUM_RUNS = 5
EPOCHS = 80
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 5e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.backends.cudnn.benchmark = True
print(f"Using device: {DEVICE}")

# Load Dataset

In [ ]:
path = kagglehub.dataset_download("orvile/bus-bra-a-breast-ultrasound-dataset")
import glob
csv_candidates = glob.glob(os.path.join(path, '**', '*.csv'), recursive=True)
img_candidates = glob.glob(os.path.join(path, '**', 'Images'), recursive=True)
mask_candidates = glob.glob(os.path.join(path, '**', 'Masks'), recursive=True)
csv_path = csv_candidates[0]
images_root = img_candidates[0]
masks_root = mask_candidates[0]

df_meta = pd.read_csv(csv_path)
entries = []
for _, row in df_meta.iterrows():
    bid = str(row['ID'])
    img_f = os.path.join(images_root, f"{bid}.png")
    mask_f = os.path.join(masks_root, bid.replace('bus_', 'mask_') + '.png')
    if os.path.exists(img_f) and os.path.exists(mask_f):
        entries.append((row['Pathology'], img_f, mask_f))

df = pd.DataFrame(entries, columns=['label', 'image_path', 'mask_path'])
stratify = df['label'] if df['label'].value_counts().min() >= 2 else None
train_df, val_df = train_test_split(df, test_size=0.2, stratify=stratify)
print(f"Found {len(df)} pairs. Train: {len(train_df)}, Val: {len(val_df)}")

In [ ]:
train_transform = A.Compose([
    A.Resize(*IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(0.05, 0.1, 15, p=0.5),
    A.ElasticTransform(alpha=50, sigma=50*0.05, p=0.3),
    A.RandomBrightnessContrast(p=0.8),
    A.GaussNoise(p=0.3),
    A.Normalize(mean=(0.0, 0.0, 0.0), std=(1.0, 1.0, 1.0), max_pixel_value=255.0),
    ToTensorV2(),
])
val_transform = A.Compose([
    A.Resize(*IMG_SIZE),
    A.Normalize(mean=(0.0, 0.0, 0.0), std=(1.0, 1.0, 1.0), max_pixel_value=255.0),
    ToTensorV2(),
])

In [ ]:
train_ds = BusbraDataset(train_df, IMG_SIZE, train_transform)
val_ds = BusbraDataset(val_df, IMG_SIZE, val_transform)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
print("Data ready.")

# Model Training (5 runs)

In [ ]:
criterion = BCEDiceLoss(weight_bce=0.5, weight_dice=0.5).to(DEVICE)
all_histories, best_ious, best_dices, best_precs = [], [], [], []

for run in range(1, NUM_RUNS + 1):
    print(f"Run {run}/{NUM_RUNS}".center(50, "-"))
    model = BaseUNet(in_ch=3, out_ch=1).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE,
                           weight_decay=WEIGHT_DECAY)
    scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='max',
                                               factor=0.1, patience=3)

    history, run_best_iou, run_best_dice, run_best_prec = train(
        model, DEVICE, train_loader, val_loader, criterion,
        optimizer, scheduler, num_epochs=EPOCHS,
        save_path=f'best_unet_baseline_run{run}.pth'
    )

    all_histories.append(history)
    best_ious.append(run_best_iou)
    best_dices.append(run_best_dice)
    best_precs.append(run_best_prec)
    print(f"Run {run} best -> IoU: {run_best_iou:.4f}, "
          f"Dice: {run_best_dice:.4f}, Prec: {run_best_prec:.4f}")

    predict_compare(model, DEVICE, val_loader, num_samples=5)

print(f"\nAll runs done.")

# Results

In [ ]:
dice_arr = np.array(best_dices)
prec_arr = np.array(best_precs)
iou_arr = np.array(best_ious)

print(f"Base U-Net on BUS-BRA (5 runs):")
print(f"  Dice      : {dice_arr.mean()*100:.2f} +/- {dice_arr.std()*100:.2f} %")
print(f"  Precision : {prec_arr.mean()*100:.2f} +/- {prec_arr.std()*100:.2f} %")
print(f"  mIoU      : {iou_arr.mean()*100:.2f} +/- {iou_arr.std()*100:.2f} %")

In [ ]:
params_m = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Params: {params_m:.2f}M")

# Inference Time

In [ ]:
model.eval()
dummy = torch.randn(1, 3, 256, 256, device=DEVICE)
with torch.no_grad():
    for _ in range(10): _ = model(dummy)
times = []
with torch.no_grad():
    for _ in range(100):
        if DEVICE.type == 'cuda': torch.cuda.synchronize()
        t0 = time.time()
        _ = model(dummy)
        if DEVICE.type == 'cuda': torch.cuda.synchronize()
        times.append(time.time() - t0)
times = np.array(times)
print(f"Inference: {times.mean()*1000:.2f} +/- {times.std()*1000:.2f} ms/image")